# Tata Steel AI Hackathon — XGBoost Baseline

Binary classification on steel coil features (`X1`–`X49`) with stratified 5-fold CV, imbalance handling, and accuracy-focused threshold tuning.

**Outputs** (saved under `models/xgboost-baseline/outputs/runs/<timestamp>/`):
- `metrics.json`, `run_config.json`, `oof_predictions.csv`
- `plots/` — PR curve, threshold sweep, confusion matrix, feature importance
- `artifacts/` — model + imputer
- `predictions/submission.csv`

## 1. Setup

In [ ]:
%pip install -q xgboost scikit-learn pandas numpy joblib matplotlib

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, average_precision_score, classification_report
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

print("python:", sys.version.split()[0])
print("xgboost:", xgb.__version__)
print("sklearn:", sklearn.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", plt.matplotlib.__version__)

## 2. Paths & run directory

Local runs write into this repo. Colab runs should point `REPO_ROOT` at your Drive clone.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
METHOD_DIR = Path(".").resolve()

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/drive/MyDrive/tata-steel-ai-hackathon")
    DATA_DIR = REPO_ROOT / "dataset"
    METHOD_DIR = REPO_ROOT / "models" / "xgboost-baseline"
else:
    REPO_ROOT = Path("../..").resolve()
    DATA_DIR = REPO_ROOT / "dataset"
    METHOD_DIR = REPO_ROOT / "models" / "xgboost-baseline"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.plotting import (
    plot_confusion_matrix,
    plot_feature_importance,
    plot_fold_scores,
    plot_pr_curve,
    plot_threshold_sweep,
)
from utils.run_artifacts import copy_to_latest_summary, create_run_dir, save_metrics, save_run_config

RUN_DIR = create_run_dir(METHOD_DIR)
ARTIFACTS_DIR = RUN_DIR / "artifacts"
PLOTS_DIR = RUN_DIR / "plots"
PREDICTIONS_DIR = RUN_DIR / "predictions"

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RUN_DIR:", RUN_DIR)

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

FEATURES = [c for c in train.columns if c.startswith("X")]
assert len(FEATURES) == 49
assert len(set(train["CoilID"]) & set(test["CoilID"])) == 0

print("train:", train.shape, " test:", test.shape)
print(train["Y"].value_counts(normalize=True).rename("rate"))

## 3. Reproducibility

In [ ]:
RANDOM_STATE = 42
N_SPLITS = 5
np.random.seed(RANDOM_STATE)

## 4. Cross-validation

In [ ]:
X = train[FEATURES].values
y = train["Y"].astype(int).values
coil_ids = train["CoilID"].values

scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
majority_baseline = float((y == 0).mean())

xgb_params = dict(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_proba = np.zeros(len(y))
fold_pr_aucs = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    imp = SimpleImputer(strategy="median")
    X_tr = imp.fit_transform(X[tr_idx])
    X_va = imp.transform(X[va_idx])
    model = XGBClassifier(**xgb_params)
    model.fit(X_tr, y[tr_idx], verbose=False)
    oof_proba[va_idx] = model.predict_proba(X_va)[:, 1]
    fold_pr = average_precision_score(y[va_idx], oof_proba[va_idx])
    fold_pr_aucs.append(float(fold_pr))
    print(f"fold {fold} PR-AUC: {fold_pr:.4f}")

## 5. Threshold tuning & metrics

In [ ]:
best_t, best_acc = 0.5, 0.0
for t in np.linspace(0.01, 0.99, 99):
    pred = (oof_proba >= t).astype(int)
    acc = accuracy_score(y, pred)
    if acc > best_acc:
        best_t, best_acc = float(t), float(acc)

oof_pr_auc = float(average_precision_score(y, oof_proba))
oof_pred = (oof_proba >= best_t).astype(int)

print(f"OOF PR-AUC: {oof_pr_auc:.4f}")
print(f"OOF accuracy @ threshold={best_t:.3f}: {best_acc:.4f}")
print(f"vs majority baseline: {majority_baseline:.4f}")
print()
print(classification_report(y, oof_pred, digits=4))

## 6. Save OOF predictions & plots

In [ ]:
pd.DataFrame(
    {"CoilID": coil_ids, "y_true": y, "oof_proba": oof_proba, "oof_pred": oof_pred}
).to_csv(RUN_DIR / "oof_predictions.csv", index=False)

plot_pr_curve(y, oof_proba, PLOTS_DIR / "oof_pr_curve.png", "OOF precision-recall")
plot_threshold_sweep(
    y, oof_proba, PLOTS_DIR / "threshold_sweep.png",
    best_threshold=best_t, majority_baseline=majority_baseline,
)
plot_confusion_matrix(y, oof_pred, PLOTS_DIR / "confusion_matrix_oof.png", f"OOF confusion matrix (t={best_t:.3f})")
plot_fold_scores({"PR-AUC": fold_pr_aucs}, PLOTS_DIR / "fold_pr_auc.png", "CV fold PR-AUC")

print("Plots saved to:", PLOTS_DIR)

## 7. Train final model & predict test

In [ ]:
import joblib

imp_full = SimpleImputer(strategy="median")
X_full = imp_full.fit_transform(X)
X_test = imp_full.transform(test[FEATURES].values)

final_model = XGBClassifier(**xgb_params)
final_model.fit(X_full, y, verbose=False)

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_t).astype(int)

final_model.save_model(str(ARTIFACTS_DIR / "xgb_model.json"))
joblib.dump(final_model, ARTIFACTS_DIR / "model.joblib")
joblib.dump(imp_full, ARTIFACTS_DIR / "imputer.joblib")
joblib.dump(
    {"method": "xgboost-baseline", "threshold": best_t, "scale_pos_weight": float(scale_pos_weight), "features": FEATURES},
    ARTIFACTS_DIR / "meta.joblib",
)

from utils.run_artifacts import write_artifacts_manifest, print_saved_artifacts
write_artifacts_manifest(ARTIFACTS_DIR)
print_saved_artifacts(ARTIFACTS_DIR)

plot_feature_importance(FEATURES, final_model.feature_importances_, PLOTS_DIR / "feature_importance.png")
print(f"Test positive predictions: {(test_pred == 1).sum()} / {len(test_pred)}")

## 8. Save metrics, submission, and latest summary

In [ ]:
submission = pd.DataFrame({"CoilID": test["CoilID"], "Y": test_pred.astype(int)})
submission_path = PREDICTIONS_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

pd.DataFrame({"CoilID": test["CoilID"], "proba": test_proba, "Y": test_pred}).to_csv(
    PREDICTIONS_DIR / "test_predictions.csv", index=False
)

metrics = {
    "method": "xgboost-baseline",
    "oof_pr_auc": oof_pr_auc,
    "oof_accuracy": best_acc,
    "best_threshold": best_t,
    "majority_baseline_accuracy": majority_baseline,
    "fold_pr_auc": fold_pr_aucs,
    "test_positives": int((test_pred == 1).sum()),
    "train_rows": len(y),
    "positive_rate": float(y.mean()),
}
save_metrics(RUN_DIR, metrics)
save_run_config(RUN_DIR, {"data_dir": str(DATA_DIR), "hyperparameters": xgb_params})
copy_to_latest_summary(METHOD_DIR, RUN_DIR)

print("Run directory:", RUN_DIR)
print("Latest summary:", METHOD_DIR / "outputs" / "latest")
print("Submission:", submission_path)
submission.head()

## 9. Display plots inline

In [ ]:
for name in ["oof_pr_curve.png", "threshold_sweep.png", "confusion_matrix_oof.png", "feature_importance.png", "fold_pr_auc.png"]:
    path = PLOTS_DIR / name
    if path.is_file():
        img = plt.imread(path)
        plt.figure(figsize=(8, 5))
        plt.imshow(img)
        plt.axis("off")
        plt.title(name)
        plt.show()

## 10. Download submission (Colab only)

In [ ]:
if IN_COLAB:
    from google.colab import files

    files.download(str(submission_path))
else:
    print(f"Validate with:\npython .cursor/skills/tata-steel-submission/scripts/validate_submission.py {submission_path}")